In [ ]:

import pandas as pd
import numpy as np

from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ============================================================
# CONFIG
# ============================================================

CSV_PATH = "dataset_clean_v2.csv"
TARGET = "orders_total"
DATE_COLUMN = "date"
TEST_DAYS = 20
RIDGE_ALPHA = 10.0   # regularization strength - higher = simpler/more stable model

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(CSV_PATH)
df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN])
df = df.sort_values(DATE_COLUMN).reset_index(drop=True)

print("=" * 70)
print("DATASET")
print("=" * 70)
print(f"Rows:  {len(df)}")
print(f"Start: {df[DATE_COLUMN].min().date()}")
print(f"End:   {df[DATE_COLUMN].max().date()}")

# ============================================================
# CREATE ORDER LAG FEATURES
# ============================================================

for lag in [1, 2, 3, 7, 14]:
    df[f"orders_lag_{lag}"] = df[TARGET].shift(lag)

df["orders_roll_7"] = df[TARGET].shift(1).rolling(window=7).mean()
df["orders_roll_14"] = df[TARGET].shift(1).rolling(window=14).mean()

# ============================================================
# CALENDAR FEATURES
# ============================================================

df["day_of_week"] = df[DATE_COLUMN].dt.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 4).astype(int)

weekday_dummies = pd.get_dummies(df["day_of_week"], prefix="weekDay", dtype=float)
for i in range(7):
    column = f"weekDay_{i}"
    if column not in weekday_dummies.columns:
        weekday_dummies[column] = 0.0
weekday_dummies = weekday_dummies[[f"weekDay_{i}" for i in range(7)]]
df = pd.concat([df, weekday_dummies], axis=1)

# NOTE: no "trend" feature anymore - see CHANGES below.

# ============================================================
# REMOVE FEATURES
# ============================================================

features_to_delete = [

    # Same-day order outcomes (leakage)
    "orders_data_available",
    "orders_completed",
    "orders_canceled",
    "orders_pending",
    "unique_order_users",
    "cancelled_orders",
    "successful_orders",
    "kabul_order_count",
    "herat_order_count",
    "online_payment_orders",
    "average_order_item_count",
    "average_order_value",
    "first_orders_count",
    "repeat_orders_count",

    # Marketing (kept lag versions instead of same-day)
    "meta_leads",
    "meta_cpc_usd",
    "meta_cpm_usd",
    "meta_active_campaigns",
    "meta_results_reported",
    "google_ads_cost_usd_estimated",
    "google_ads_avg_cpc_usd",
    "blogger_spend_afn",
    "total_ad_spend_afn",

    # Delivery / discount
    "delivery_share_lag7_avg",
    "delivery_share_lag1",
    "avg_delivery_fee_lag7_avg",
    "avg_delivery_fee_lag1",
    "avg_discount_amount_lag1",
    "avg_discount_amount_lag7_avg",

    # Restaurant / platform same-day
    "new_active_restaurants",
    "rating_sum_that_day",
    "featured_that_day",
    "commission_sum_that_day",
    "new_available_dishes",
    "count_that_day",
    "new_featured_dishes",
    "cumulative_featured_restaurants",
    "cumulative_commission_sum",
    "avg_commission_rate",

    # Blogger
    "blogger_views_reach_lag7_avg",
    "blogger_views_reach_lag1",

    # Advertising
    "spend_lag_1",
    "meta_impressions_lag1",
    "meta_impressions_lag7_avg",
    "meta_link_clicks_lag1",
    "meta_ctr_percent_lag7_avg",
    "meta_cpm_usd_lag1",
    "meta_cpc_usd_lag7_avg",

    # Google Ads
    "google_ads_ctr_percent_estimated",
    "google_ads_ctr_percent_estimated_lag1",
    "google_ads_ctr_percent_estimated_lag7_avg",

    # Item / order value
    "avg_item_price_lag1",
    "avg_item_price_lag7_avg",
    "avg_order_food_value_lag1",
    "avg_order_food_value_lag7_avg",

    # Calendar duplicate
    "day_of_week_num",

    # --- NEW: raw monotonic cumulative counters ---
    # These climb every single day almost like a clock, so they act
    # as a hidden "trend" feature and destabilize the linear model
    # (huge, unstable coefficients -> bad extrapolation on new dates).
    "cumulative_users",
    "cumulative_active_restaurants",
    "cumulative_restaurant_count",
    "cumulative_rating_sum",
    "cumulative_available_dishes",
    "cumulative_featured_dishes",
]

features_to_delete = list(set(features_to_delete))
features_to_delete = [col for col in features_to_delete if col in df.columns]
df = df.drop(columns=features_to_delete, errors="ignore")

print(f"\nDeleted {len(features_to_delete)} columns")

# ============================================================
# FEATURES
# ============================================================

numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()

if TARGET in numeric_columns:
    numeric_columns.remove(TARGET)
if "day_of_week" in numeric_columns:
    numeric_columns.remove("day_of_week")
if "Unnamed: 0" in numeric_columns:
    numeric_columns.remove("Unnamed: 0")

FEATURES = numeric_columns

print("\n" + "=" * 70)
print("FEATURES")
print("=" * 70)
print(f"Total features: {len(FEATURES)}")
for i, feature in enumerate(FEATURES, 1):
    print(f"{i}. {feature}")

# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

test_start = len(df) - TEST_DAYS
train = df.iloc[:test_start].copy()
test = df.iloc[test_start:].copy()

print("\n" + "=" * 70)
print("TRAIN / TEST SPLIT")
print("=" * 70)
print(f"Training period: {train[DATE_COLUMN].min().date()} → {train[DATE_COLUMN].max().date()}")
print(f"Testing period:  {test[DATE_COLUMN].min().date()} → {test[DATE_COLUMN].max().date()}")
print(f"Training samples: {len(train)}")
print(f"Testing samples:  {len(test)}")

# ============================================================
# REMOVE ROWS WITHOUT ENOUGH LAG HISTORY
# ============================================================

train_model = train.dropna(
    subset=["orders_lag_1", "orders_lag_2", "orders_lag_3", "orders_lag_7", "orders_lag_14"]
).copy()

X_train = train_model[FEATURES]
y_train = train_model[TARGET]

print(f"\nActual training rows used: {len(train_model)}")
print(f"Features vs. training rows ratio: {len(FEATURES)} features / {len(train_model)} rows")

# ============================================================
# MODEL
# ============================================================
# CHANGED: LinearRegression -> Ridge, and added a scaler.
# Plain LinearRegression with ~40+ correlated features and only
# ~150 rows produces huge, unstable coefficients (multicollinearity).
# Ridge shrinks the coefficients so the model generalizes instead
# of memorizing/extrapolating. StandardScaler makes the shrinkage
# penalty fair across features that live on very different scales
# (e.g. spend in dollars vs. a percentage vs. a 0-5 rating).
print(df.head())
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=RIDGE_ALPHA)),
])

# model = Pipeline([
#     (
#         "imputer",
#         SimpleImputer(strategy="median")
#     ),
#     (
#         "linear_regression",
#         LinearRegression()
#     )
# ])

model.fit(X_train, y_train)
print("\nModel trained.")

# ============================================================
# FUTURE FEATURE BUILDER (unchanged logic)
# ============================================================

history = {i: df.loc[i, TARGET] for i in range(test_start)}

def build_future_features(row_index, history):
    current_row = df.loc[row_index]
    values = {}

    for feature in FEATURES:
        if feature.startswith("orders_lag_"):
            continue
        if feature in ["orders_roll_7", "orders_roll_14"]:
            continue
        values[feature] = current_row[feature]

    for lag in [1, 2, 3, 7, 14]:
        previous_index = row_index - lag
        values[f"orders_lag_{lag}"] = history.get(previous_index, np.nan)

    previous_values_7 = [history.get(i, np.nan) for i in range(row_index - 7, row_index)]
    previous_values_7 = [v for v in previous_values_7 if pd.notna(v)]
    values["orders_roll_7"] = np.mean(previous_values_7) if previous_values_7 else np.nan

    previous_values_14 = [history.get(i, np.nan) for i in range(row_index - 14, row_index)]
    previous_values_14 = [v for v in previous_values_14 if pd.notna(v)]
    values["orders_roll_14"] = np.mean(previous_values_14) if previous_values_14 else np.nan

    return pd.DataFrame([values], columns=FEATURES)

# ============================================================
# PREDICTIONS: recursive AND one-step-ahead (both reported)
# ============================================================
# CHANGED: added a "one-step-ahead" pass alongside the original
# recursive forecast, so you can see how much of the error comes
# from the model itself vs. from errors compounding across the
# 10-day recursive horizon.

predictions_recursive = []
predictions_one_step = []

for i in range(test_start, len(df)):

    # --- recursive: feeds back PREDICTED history (original approach) ---
    X_future_recursive = build_future_features(i, history)
    pred_recursive = max(0, model.predict(X_future_recursive)[0])
    predictions_recursive.append(pred_recursive)
    history[i] = pred_recursive  # compounds forward

    # --- one-step-ahead: always uses ACTUAL history ---
    actual_history = {j: df.loc[j, TARGET] for j in range(i)}
    X_future_actual = build_future_features(i, actual_history)
    pred_one_step = max(0, model.predict(X_future_actual)[0])
    predictions_one_step.append(pred_one_step)

# ============================================================
# RESULTS
# ============================================================

results = test[[DATE_COLUMN, TARGET]].copy()
results["predicted_recursive"] = predictions_recursive
results["predicted_one_step"] = predictions_one_step

for col in ["predicted_recursive", "predicted_one_step"]:
    results[f"abs_error_{col}"] = (results[col] - results[TARGET]).abs()

def summarize(name, pred_col, err_col):
    mae = mean_absolute_error(results[TARGET], results[pred_col])
    rmse = np.sqrt(mean_squared_error(results[TARGET], results[pred_col]))
    r2 = r2_score(results[TARGET], results[pred_col])
    non_zero = results[TARGET] != 0
    mape = (results.loc[non_zero, err_col] / results.loc[non_zero, TARGET].abs()).mean() * 100
    acc = np.clip(100 - mape, 0, 100)
    print(f"\n{name}")
    print(f"  Accuracy (100-MAPE): {acc:.2f}%")
    print(f"  MAPE : {mape:.2f}%")
    print(f"  MAE  : {mae:.2f} orders")
    print(f"  RMSE : {rmse:.2f}")
    print(f"  R2   : {r2:.4f}")

print("\n" + "=" * 90)
print("DAILY PREDICTIONS")
print("=" * 90)
print(results.to_string(index=False))

print("\n" + "=" * 90)
print("MODEL PERFORMANCE")
print("=" * 90)
summarize("RECURSIVE (original approach - predicted values feed forward)",
          "predicted_recursive", "abs_error_predicted_recursive")
summarize("ONE-STEP-AHEAD (each day predicted from real history)",
          "predicted_one_step", "abs_error_predicted_one_step")
